# Trizzy Writer ? WAN 2.1 Colab Render Worker

Reusable Google Colab GPU worker for **WAN 2.1 T2V 1.3B**. It reads JSON render jobs from Google Drive, renders MP4 files, and writes machine-readable progress/status files for Trizzy Writer.

**Runtime:** In Colab choose `Runtime ? Change runtime type ? GPU`, then run the cells in order.


## 1. Install pinned runtime dependencies

The notebook uses Hugging Face Diffusers' official WAN pipeline. The first run installs packages and may require a runtime restart if Colab already loaded conflicting versions.


In [ ]:
%pip install -q --upgrade \
  "diffusers==0.39.0" \
  "transformers>=4.53,<5" \
  "accelerate>=1.8,<2" \
  "huggingface_hub>=0.33,<2" \
  "sentencepiece>=0.2,<1" \
  "ftfy>=6.3,<7" \
  "imageio>=2.37,<3" \
  "imageio-ffmpeg>=0.6,<1"


## 2. Validate GPU


In [ ]:
import os, sys, json, time, shutil, traceback
from pathlib import Path
import torch

assert torch.cuda.is_available(), "No GPU detected. In Colab select Runtime > Change runtime type > GPU."
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {GPU_NAME}")
print(f"VRAM: {VRAM_GB:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print("WAN 2.1 1.3B worker can continue.")


## 3. Mount Drive and create persistent worker folders


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/TrizzyWriterVideo')
QUEUE = ROOT / 'queue'
PROCESSING = ROOT / 'processing'
OUTPUTS = ROOT / 'outputs'
FAILED = ROOT / 'failed'
STATUS = ROOT / 'status'
MODEL_CACHE = ROOT / 'model-cache'

for folder in (QUEUE, PROCESSING, OUTPUTS, FAILED, STATUS, MODEL_CACHE):
    folder.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(MODEL_CACHE)
os.environ['HF_HUB_CACHE'] = str(MODEL_CACHE / 'hub')
print(f"Worker root: {ROOT}")


## 4. Load WAN 2.1 T2V 1.3B

The checkpoint is cached in Google Drive after the first download. CPU offload lowers peak VRAM use, which improves compatibility with common Colab GPUs.


In [ ]:
from diffusers import AutoModel, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

MODEL_ID = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'

vae = AutoModel.from_pretrained(
    MODEL_ID,
    subfolder='vae',
    torch_dtype=torch.float32,
    cache_dir=str(MODEL_CACHE),
)

pipe = WanPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    torch_dtype=torch.bfloat16,
    cache_dir=str(MODEL_CACHE),
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
print(f"Loaded {MODEL_ID}")


## 5. Render-job format

Place one JSON file per render in the Drive `queue` folder. Frame counts are normalized to WAN's required `4k + 1` pattern. The initial worker targets stable 480p-class output.


In [ ]:
DEFAULT_JOB = {
    "id": "neon-rain-test",
    "task": "text-to-video",
    "prompt": "A cinematic night scene of a confident male R&B artist walking through neon rain, realistic lighting, slow dolly camera, premium music video, detailed face, natural movement",
    "negative_prompt": "blurry, low quality, distorted face, deformed hands, extra limbs, duplicate person, text, watermark, static frame",
    "width": 832,
    "height": 480,
    "num_frames": 81,
    "fps": 16,
    "steps": 30,
    "guidance_scale": 5.0,
    "seed": 4231993
}
print(json.dumps(DEFAULT_JOB, indent=2))


## 6. Worker implementation


In [ ]:
def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    temp.replace(path)


def normalize_job(raw: dict) -> dict:
    job = {**DEFAULT_JOB, **raw}
    job_id = str(job.get('id') or f"render-{int(time.time())}")
    job['id'] = ''.join(c for c in job_id if c.isalnum() or c in ('-', '_'))[:80]
    if job.get('task') != 'text-to-video':
        raise ValueError("This stage supports task='text-to-video' only.")
    if not str(job.get('prompt', '')).strip():
        raise ValueError('A non-empty prompt is required.')

    # Stable 480p presets; dimensions must be divisible by 16.
    width = max(256, min(832, int(job.get('width', 832))))
    height = max(256, min(480, int(job.get('height', 480))))
    job['width'] = width - (width % 16)
    job['height'] = height - (height % 16)

    requested_frames = max(17, min(121, int(job.get('num_frames', 81))))
    job['num_frames'] = ((requested_frames - 1) // 4) * 4 + 1
    job['fps'] = max(8, min(30, int(job.get('fps', 16))))
    job['steps'] = max(10, min(50, int(job.get('steps', 30))))
    job['guidance_scale'] = max(1.0, min(10.0, float(job.get('guidance_scale', 5.0))))
    job['seed'] = int(job.get('seed', 0))
    return job


def render_job(job: dict) -> Path:
    job = normalize_job(job)
    job_id = job['id']
    status_path = STATUS / f'{job_id}.json'
    output_path = OUTPUTS / f'{job_id}.mp4'
    started = time.time()

    write_json(status_path, {
        'id': job_id,
        'state': 'rendering',
        'progress': 5,
        'started_at': started,
        'settings': job,
        'gpu': GPU_NAME,
    })

    generator = torch.Generator(device='cuda').manual_seed(job['seed'])

    def callback(pipe_obj, step_index, timestep, callback_kwargs):
        progress = 5 + int(90 * (step_index + 1) / job['steps'])
        write_json(status_path, {
            'id': job_id,
            'state': 'rendering',
            'progress': min(progress, 95),
            'step': step_index + 1,
            'total_steps': job['steps'],
            'started_at': started,
            'gpu': GPU_NAME,
        })
        return callback_kwargs

    result = pipe(
        prompt=job['prompt'],
        negative_prompt=job.get('negative_prompt') or None,
        width=job['width'],
        height=job['height'],
        num_frames=job['num_frames'],
        num_inference_steps=job['steps'],
        guidance_scale=job['guidance_scale'],
        generator=generator,
        callback_on_step_end=callback,
    )
    frames = result.frames[0]
    export_to_video(frames, str(output_path), fps=job['fps'])

    finished = time.time()
    write_json(status_path, {
        'id': job_id,
        'state': 'complete',
        'progress': 100,
        'output': str(output_path),
        'duration_seconds': round(finished - started, 2),
        'started_at': started,
        'finished_at': finished,
        'gpu': GPU_NAME,
        'settings': job,
    })
    torch.cuda.empty_cache()
    return output_path


def process_next_job():
    jobs = sorted(QUEUE.glob('*.json'), key=lambda p: p.stat().st_mtime)
    if not jobs:
        print('No queued jobs.')
        return None

    source = jobs[0]
    processing_path = PROCESSING / source.name
    shutil.move(str(source), str(processing_path))
    raw = json.loads(processing_path.read_text(encoding='utf-8'))
    job_id = str(raw.get('id') or processing_path.stem)

    try:
        output = render_job(raw)
        processing_path.unlink(missing_ok=True)
        print(f'Complete: {output}')
        return output
    except Exception as exc:
        error_text = traceback.format_exc()
        failed_path = FAILED / processing_path.name
        shutil.move(str(processing_path), str(failed_path))
        write_json(STATUS / f'{job_id}.json', {
            'id': job_id,
            'state': 'failed',
            'progress': 0,
            'error': str(exc),
            'traceback': error_text,
            'failed_job': str(failed_path),
            'gpu': GPU_NAME,
        })
        print(error_text)
        raise

print('Worker functions loaded.')


## 7. Create a test queue job


In [ ]:
test_path = QUEUE / f"{DEFAULT_JOB['id']}.json"
write_json(test_path, DEFAULT_JOB)
print(f"Queued: {test_path}")


## 8. Process one job

Run this cell to render the oldest queued request.


In [ ]:
output_path = process_next_job()
output_path


## 9. Continuous worker mode

This loop checks Drive for new jobs every 15 seconds. Stop the cell when you are finished rendering. Colab sessions are temporary, so the worker only runs while the notebook session remains connected.


In [ ]:
POLL_SECONDS = 15
print(f'Watching {QUEUE}. Stop this cell to disconnect the worker.')
try:
    while True:
        if list(QUEUE.glob('*.json')):
            process_next_job()
        else:
            time.sleep(POLL_SECONDS)
except KeyboardInterrupt:
    print('Worker stopped.')


## Current capability

- Working WAN 2.1 1.3B text-to-video pipeline
- Persistent model cache in Google Drive
- JSON queue and processing folders
- MP4 output generation
- Per-job progress, completion, and failure status JSON
- Deterministic seeds and validated render settings

The next stage is connecting Trizzy Writer's web interface to create queue jobs and read statuses/results.
